# Kaggriculture JAX 環境 — GPU ベンチマーク & 忠実性検証

**Runtime → Change runtime type → T4 GPU** を選んでから、上から順に実行してください。

このノートブックは2つを行います:
1. **忠実性** — GPU 上の状態ダイジェストが、CPU で公式 engine と一致検証済みの値と合うか
2. **高速化** — batch を振って env-steps/s を測定

所要 5〜10分。Colab には JAX+CUDA がプリインストール済みなので追加DLは不要です。

In [ ]:
import jax
print('JAX', jax.__version__)
print('devices:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'GPU ランタイムを選択してください (Runtime > Change runtime type > T4 GPU)'

In [ ]:
# 環境コードを取得 (state.py / env.py のみ使用。src/ への依存なし)
!git clone -q --depth 1 -b feature-simulator-v2 https://github.com/YuriNakayama/kaggriculture /content/kg
import sys, shutil, pathlib
src = pathlib.Path('/content/kg/backend/src/simulate/jaxenv')
dst = pathlib.Path('/content/jb'); dst.mkdir(exist_ok=True)
for f in ('state.py','env.py'):
    t = (src/f).read_text().replace('from .state import','from state import')
    (dst/f).write_text(t)
sys.path.insert(0,'/content/jb')
print('ready:', sorted(p.name for p in dst.iterdir()))

## 1. 忠実性 — GPU 上の状態が CPU 検証済みの値と一致するか

固定 action 列を15日ぶん流し、12個の状態配列の SHA256 を取ります。
CPU 側 (公式 engine と seed 100本で完全一致を確認済み) の期待値と突き合わせます。

In [ ]:
import hashlib, jax, jax.numpy as jnp
import env as E
from state import initial_state, MAX_UNITS

STEP = E.make_step(weed_chance=0.0, shop_sell_interval=4,
                   center_sell_interval=24, shop_unlock_interval=10**6)
B, DAYS = 8, 15
rng = jax.random.PRNGKey(0)
s = initial_state(B, seed=0)
for n in range(DAYS*24):
    k = jax.random.fold_in(rng, n)
    us, ms = (B,2,MAX_UNITS), (B,2,E.MAX_MARKET_ORDERS)
    hour = n % 24
    loop_op = {1:E.OP_PLANT,2:E.OP_WATER,3:E.OP_WATER,5:E.OP_HARVEST,7:E.OP_DROP}.get(hour,E.OP_PASS)
    op  = jax.random.randint(jax.random.fold_in(k,1), us, 0, E.N_OPS).at[:,:,0].set(loop_op)
    arg = jax.random.randint(jax.random.fold_in(k,2), us, 0, 12).at[:,:,0].set(0)
    qty = jax.random.randint(jax.random.fold_in(k,3), us, 0, 4)
    mop = jax.random.randint(jax.random.fold_in(k,4), ms, 0, 7)
    mop = mop.at[:,:,0].set(E.MARKET_BUY_SEED if hour==0 else (E.MARKET_SELL if hour==8 else E.MARKET_NONE))
    mit = jax.random.randint(jax.random.fold_in(k,5), ms, 0, 9)
    mqt = jax.random.randint(jax.random.fold_in(k,6), ms, 0, 4)
    s = STEP(s, op, arg, qty, mop, mit, mqt)

EXPECTED = {'animal':'e202c0cf84a7cd41','animal_shed':'3158570784fcab36','carried':'2d07a41ae9927700',
 'crop':'ee506e29a41b3a8b','kind':'5c22a5e65fe57d08','lands_bought':'3110ab1939d07217',
 'market_inv':'7cf5da2bd2566757','money':'2b4ff02479a20654','seeds':'546dba8e52359fd4',
 'shed':'460cb091b241731c','unit_active':'2435b3bb50cad942','yield_units':'41cc89efb05c356d'}

bad = []
for name, want in EXPECTED.items():
    got = hashlib.sha256(jax.device_get(getattr(s,name)).tobytes()).hexdigest()[:16]
    mark = 'OK ' if got == want else 'MISMATCH'
    if got != want: bad.append(name)
    print(f'{mark} {name:16s} {got}')
print()
print('=== 忠実性: GPU と CPU 検証済みの値が完全一致 ===' if not bad else f'=== 不一致: {bad} ===')

## 2. 高速化 — batch 別スループット

In [ ]:
import time
def rollout_fn(B):
    def body(i, s):
        hour = jnp.remainder(i,24)
        op = jnp.where(hour==1,E.OP_PLANT,jnp.where((hour==2)|(hour==3),E.OP_WATER,
             jnp.where(hour==5,E.OP_HARVEST,jnp.where(hour==7,E.OP_DROP,E.OP_PASS))))
        mop = jnp.where(hour==0,E.MARKET_BUY_SEED,jnp.where(hour==8,E.MARKET_SELL,E.MARKET_NONE))
        q = jnp.where(hour==0,1,jnp.where(hour==8,5,0))
        us, ms = (B,2,MAX_UNITS), (B,2,E.MAX_MARKET_ORDERS)
        z = jnp.zeros(us, jnp.int32)
        return STEP(s, z.at[:,:,0].set(op), z, z,
                    jnp.zeros(ms,jnp.int32).at[:,:,0].set(mop),
                    jnp.zeros(ms,jnp.int32),
                    jnp.zeros(ms,jnp.int32).at[:,:,0].set(q))
    return jax.jit(lambda s,n: jax.lax.fori_loop(0,n,body,s))

STEPS = 720   # 1 シーズン
OFFICIAL = 768  # 公式 env.run の env-steps/s (ローカル実測)
print(f'{"batch":>7} {"wall(s)":>9} {"compile(s)":>11} {"env-steps/s":>14} {"vs official":>12}')
for B in [1, 64, 1024, 8192, 65536]:
    try:
        f, s0, n = rollout_fn(B), initial_state(B, seed=0), jnp.int32(STEPS)
        t=time.perf_counter(); jax.block_until_ready(f(s0,n)); comp=time.perf_counter()-t
        t=time.perf_counter(); jax.block_until_ready(f(s0,n)); wall=time.perf_counter()-t
        sps = B*STEPS/wall
        print(f'{B:>7} {wall:>9.3f} {comp:>11.1f} {sps:>14,.0f} {sps/OFFICIAL:>11,.0f}x')
    except Exception as e:
        print(f'{B:>7}  FAILED: {type(e).__name__}: {str(e)[:60]}'); break